In [1]:
import psutil, platform

print("CPU cores:", psutil.cpu_count(logical=True))
print("RAM (GB):", round(psutil.virtual_memory().total / 1e9, 2))
print("System:", platform.platform())


CPU cores: 8
RAM (GB): 16.89
System: Windows-10-10.0.26100-SP0


In [5]:
import gzip, json
RAW_JSONL = "data/raw/yt_metadata_en.jsonl.gz"

with gzip.open(RAW_JSONL, "rt", encoding="utf-8") as f:
    for i, line in enumerate(f):
        d = json.loads(line)
        tags = d.get("tags")
        if tags:
            print(type(tags), tags[:3] if isinstance(tags, list) else tags)
            if i >= 5: break


<class 'str'> lego city,lego police,lego city police,lego city episodes,videos de lego city,lego policia,lego bomberos,lego fire truck,lego firetruck,lego police chase,lego robbers,lego cartoons,lego movies,lego videos for kids
<class 'str'> Lego superheroes,lego hulk,hulk smash,lego marverl superheroes,lego marvel super heroes movie,lego ironman,lego spiderman,spider-man,lego heroes,lego avengers,lego captain america
<class 'str'> lego city,lego police,lego city police,lego fire truck,lego cartoons,lego for kids,lego episodes,lego polizi,lego policias,lego police chase,lego policias para niños
<class 'str'> Lego harry potter,new harry potter,harry potter new movie,movie harry potter,movie harry potter lego,lego harry,lego movies,lego cartoons
<class 'str'> lego city,lego police,lego city police,lego fire truck,lego cartoons,cartoons about lego,lego for kids,lego city movie,videos de lego city,lego policias,lego bomberos,lego polizi,polizi lego,lego para niños,lego fireman
<class 'str'

In [ ]:
import gzip, json, re, math
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
from tqdm import tqdm

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sentence_transformers import SentenceTransformer
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score

# --- tag cleaner ---
STOP = set(stopwords.words('english'))
LEMM = WordNetLemmatizer()
PUN_NUM = re.compile(r'[^a-z\s]+')

def clean_tags(x):
    """Clean a comma- or pipe-separated tag string or list."""
    if x is None:
        return ''
    # unify everything into a plain string of space-separated tags
    if isinstance(x, list):
        s = ' '.join(map(str, x))
    else:
        s = str(x).replace('|', ' ').replace(',', ' ')
    s = s.lower()
    s = PUN_NUM.sub(' ', s)           # remove punctuation/numbers
    toks = [w for w in s.split() if w not in STOP and len(w) > 1]
    toks = [LEMM.lemmatize(w) for w in toks]
    return ' '.join(toks)

# --- shannon entropy ---
def shannon_entropy(counts: np.ndarray) -> float:
    total = counts.sum()
    if total <= 0: return 0.0
    p = counts / total
    p = p[p > 0]
    return float(-(p * np.log(p)).sum())


c:\Users\Esabella D\Desktop\ada-2025-project-in5ight\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1) Pilot sample (train K-means & choose K)

In [4]:
RAW_JSONL = "data/raw/yt_metadata_en.jsonl.gz"
N_PILOT   = 150_000          # adjust based on RAM (100k–300k is fine)
BATCH_EMB = 256              # embedding batch size
model = SentenceTransformer('all-MiniLM-L6-v2')

pilot_tags = []
pilot_ids  = []

with gzip.open(RAW_JSONL, "rt", encoding="utf-8") as f:
    for i, line in enumerate(f):
        d = json.loads(line)
        tags = d.get("tags")
        if tags is None: 
            continue
        s = clean_tags(tags)
        if not s: 
            continue
        pilot_tags.append(s)
        pilot_ids.append((d.get("display_id"), d.get("channel_id")))
        if len(pilot_tags) >= N_PILOT:
            break

# Embed pilot in batches
def batched(iterable, n):
    for i in range(0, len(iterable), n):
        yield iterable[i:i+n]

emb_list = []
for chunk in tqdm(list(batched(pilot_tags, BATCH_EMB)), desc="Embed pilot"):
    emb = model.encode(chunk, batch_size=BATCH_EMB, show_progress_bar=False, normalize_embeddings=True)
    emb_list.append(emb)
pilot_emb = np.vstack(emb_list)
pilot_emb.shape


Embed pilot:  11%|█▏        | 66/586 [04:31<35:41,  4.12s/it]


KeyboardInterrupt: 

Pick K quickly (on a small subsample) and fit MiniBatchKMeans:

In [ ]:
# quick K search on a 50k subsample
idx = np.random.choice(len(pilot_emb), size=min(50_000, len(pilot_emb)), replace=False)
Xs  = pilot_emb[idx]

candidate_K = [20, 30, 40] 
scores = {}
for K in candidate_K:
    km = MiniBatchKMeans(n_clusters=K, random_state=42, batch_size=2048, n_init='auto')
    km.fit(Xs)
    # silhouette on a smaller slice for speed
    s_idx = np.random.choice(len(Xs), size=min(10_000, len(Xs)), replace=False)
    sil = silhouette_score(Xs[s_idx], km.predict(Xs[s_idx]))
    scores[K] = sil
print("Silhouette:", scores)

best_K = max(scores, key=scores.get)
print("Chosen K =", best_K)

# Fit final MiniBatchKMeans on the full pilot embeddings (still fast)
kmeans = MiniBatchKMeans(n_clusters=best_K, random_state=42, batch_size=4096, n_init='auto')
kmeans.fit(pilot_emb)


2) Full streaming pass (predict & aggregate per channel)

In [ ]:
channel_cluster_counts: dict[str, Counter] = defaultdict(Counter)
total_processed = 0

with gzip.open(RAW_JSONL, "rt", encoding="utf-8") as f:
    buf_tags, buf_chids = [], []
    for line in f:
        d = json.loads(line)
        tags = d.get("tags")
        ch   = d.get("channel_id")
        if tags is None or ch is None:
            continue
        s = clean_tags(tags)
        if not s:
            continue
        buf_tags.append(s); buf_chids.append(str(ch))

        # process in embedding batches
        if len(buf_tags) >= 5000:  # tune this for your RAM/CPU
            emb = model.encode(buf_tags, batch_size=BATCH_EMB, show_progress_bar=False, normalize_embeddings=True)
            labels = kmeans.predict(emb)
            for cid, lab in zip(buf_chids, labels):
                channel_cluster_counts[cid][int(lab)] += 1
            total_processed += len(buf_tags)
            buf_tags.clear(); buf_chids.clear()

    # flush remainder
    if buf_tags:
        emb = model.encode(buf_tags, batch_size=BATCH_EMB, show_progress_bar=False, normalize_embeddings=True)
        labels = kmeans.predict(emb)
        for cid, lab in zip(buf_chids, labels):
            channel_cluster_counts[cid][int(lab)] += 1
        total_processed += len(buf_tags)

print(f"Processed {total_processed:,} videos.")


In [ ]:
# Convert to a tidy DataFrame:
rows = []
for cid, ctr in channel_cluster_counts.items():
    total = sum(ctr.values())
    for lab, n in ctr.items():
        rows.append({"channel_id": cid, "cluster": lab, "n_vids": n, "total_vids": total})
channel_cluster = pd.DataFrame(rows)
channel_cluster['prop'] = channel_cluster['n_vids'] / channel_cluster['total_vids']

# Dominant cluster + entropy
dom = (channel_cluster.sort_values(['channel_id','prop'], ascending=[True,False])
       .drop_duplicates('channel_id'))[['channel_id','cluster','prop']]
dom = dom.rename(columns={'cluster':'cluster_dom','prop':'cluster_dom_prop'})

ent = (channel_cluster.groupby('channel_id')['n_vids']
       .apply(lambda s: shannon_entropy(s.values))
       .rename('topic_entropy')
       .reset_index())

cluster_assign = (dom.merge(ent, on='channel_id')
                     .merge(channel_cluster[['channel_id','total_vids']].drop_duplicates(), on='channel_id'))
cluster_assign.head()


3) Merge with channels_final.tsv & analyze

In [ ]:
channels = pd.read_csv("data/raw/channels_final.tsv", sep="\t")
channels['channel_id']      = channels['channel_id'].astype(str)
cluster_assign['channel_id']= cluster_assign['channel_id'].astype(str)

df = (channels[['channel_id','avg_efficiency','weights','subscriber_rank_sb',
                'avg_views','avg_duration','video_count','category_cc']]
      .merge(cluster_assign, on='channel_id', how='inner'))

# (optional) filter channels with very few clustered videos
df = df[df['total_vids'] >= 10].copy()
df.head()


In [ ]:
# Quick Visual
import seaborn as sns, matplotlib.pyplot as plt
plt.figure(figsize=(12,6))
sns.violinplot(data=df, x='cluster_dom', y='avg_efficiency', inner='quartile', cut=0)
plt.xticks(rotation=90); plt.tight_layout(); plt.show()


In [ ]:
# Weighted regression:
import statsmodels.formula.api as smf
df['log_eff'] = np.log(df['avg_efficiency'].clip(lower=1e-8))
mod = smf.wls('log_eff ~ C(cluster_dom)', data=df, weights=df['weights']).fit()
print(mod.summary())